# Chapter 11: Tokens & Permissions (Reference)

## Learning Objectives

- Print the GITHUB_TOKEN operation matrix and identify the three real exceptions
- Compare the three token identities side by side
- Reproduce the -f vs -F encoding trap and see why -f silently no-ops a boolean
- State why this repo's automerge.yml fails loudly instead of silently

## Setup

The next cell sets up reproducibility and the `PRA_MODE` toggle. You should see `PRA_MODE = 'fixture'` printed by default.

In [1]:
import os
import random
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pr_automerge").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

RANDOM_STATE: int = 42
random.seed(RANDOM_STATE)

PRA_MODE = os.environ.get("PRA_MODE", "fixture")
PRA_REPO = os.environ.get("PRA_REPO", "")

if PRA_MODE == "live":
    assert PRA_REPO, "Set PRA_REPO=owner/name to run against a real repo"

print(f"PRA_MODE = {PRA_MODE!r}")

PRA_MODE = 'fixture'


## 1. The GITHUB_TOKEN Operation Matrix

The next cell prints every operation in `PERMISSION_OPERATIONS`. You should see five operations that work and three that don't, each naming what's needed instead.

In [2]:
from labs.lab_11_tokens_and_permissions import PERMISSION_OPERATIONS, check_operation

for row in PERMISSION_OPERATIONS:
    works = "yes" if row["github_token_works"] else "NO"
    needs = f" (needs: {row['needs_instead']})" if row["needs_instead"] else ""
    print(f"{row['operation']:<32} works={works}{needs}")

read_pr_metadata                 works=yes
read_pr_files_paginated          works=yes
publish_check_run                works=yes
post_pr_comment                  works=yes
patch_allow_auto_merge_setting   works=yes
read_branch_protection_detail    works=NO (needs: PAT or GitHub App with Administration access)
enable_auto_merge                works=NO (needs: PAT or GitHub App with pull_requests: write)
push_to_protected_branch_as_bot  works=NO (needs: push to an unprotected branch, or a PAT belonging to an admin)


## 2. The Three Identities

The next cell prints `TOKEN_IDENTITIES`. You should see `GITHUB_TOKEN` alone unable to enable auto-merge, while both PAT and GitHub App can.

In [3]:
from labs.lab_11_tokens_and_permissions import TOKEN_IDENTITIES

for name, props in TOKEN_IDENTITIES.items():
    print(
        f"{name:<14} lifetime={props['lifetime']!r}, can_enable_auto_merge={props['can_enable_auto_merge']}"
    )

GITHUB_TOKEN   lifetime='one workflow run', can_enable_auto_merge=False
PAT            lifetime='until revoked or expiry', can_enable_auto_merge=True
GitHub App     lifetime='~1 hour per installation token, auto-renewed', can_enable_auto_merge=True


## 3. The `-f` vs `-F` Trap

The next cell encodes `allow_auto_merge=true` both ways. You should see `-f` produce a Python `str` and `-F` produce a Python `bool` -- the exact distinction that caused a real, live-caught bug in this repo's Gate 1 build.

In [4]:
from labs.lab_11_tokens_and_permissions import encode_gh_api_field

wrong = encode_gh_api_field("-f", "true")
right = encode_gh_api_field("-F", "true")
print(f"-f -> {wrong!r} ({type(wrong).__name__})")
print(f"-F -> {right!r} ({type(right).__name__})")

-f -> 'true' (str)
-F -> True (bool)


## Takeaways & Next Steps

This notebook's takeaway is Section 3's type difference -- a silent no-op is much harder to debug than an error, which is exactly why this trap is worth memorizing.

In [5]:
print("See resources/token_permission_matrix.md for the full, unabridged comparison.")

See resources/token_permission_matrix.md for the full, unabridged comparison.


---

📖 **Reading companion:** [Chapter 11: Tokens & Permissions](../learning_modules/chapter_11_tokens_and_permissions.md)
🔬 **Try it live:** this chapter's lab is fully offline (no `gh` calls), so `PRA_MODE=live` changes nothing here — nothing to re-run against a real repo.
